# Databricks Workspace-Setup – BIS242 SS26

Dieses Notebook erstellt **User** über die Databricks SCIM API und legt die **Datenbanken** per SQL an.

**Ausführung:** Als Workspace-Admin in Databricks importieren und Zelle für Zelle ausführen.

> **Hinweis:** Im Free Workspace gibt es kein Unity Catalog. User-Verwaltung erfolgt über die SCIM API (`/api/2.0/preview/scim/v2/Users`).

## 1. Konfiguration

In [ ]:
# Workspace-URL (ohne trailing slash)
WORKSPACE_URL = "https://dbc-15d76289-5b7c.cloud.databricks.com"

# Admin-Token: entweder hier setzen oder aus dbutils lesen
# Option A: Direkt (nur für einmalige Ausführung, danach löschen!)
# ADMIN_TOKEN = "dapi..."

# Option B: Aus Databricks Secret Scope (empfohlen)
# ADMIN_TOKEN = dbutils.secrets.get(scope="admin", key="token")

# Option C: Aus dem aktuellen Notebook-Kontext (funktioniert wenn als Admin eingeloggt)
ADMIN_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

print(f"Workspace: {WORKSPACE_URL}")
print(f"Token:     {'*' * 10}...{ADMIN_TOKEN[-4:]}")

## 2. Gruppendefinition

In [ ]:
gruppen = {
    "01": [
        "selman.dogtas@stud.hs-hannover.de",
        "aaron.koch@stud.hs-hannover.de",
        "rida.noureddine@stud.hs-hannover.de",
    ],
    "02": [
        "swastika.bhusal@stud.hs-hannover.de",
        "walaa.darwish@stud.hs-hannover.de",
        "katrin-omar.mohammad@stud.hs-hannover.de",
    ],
    "03": [],
    "04": [
        "sebastian.brand@stud.hs-hannover.de",
        "jan-henrik.weinhold@stud.hs-hannover.de",
    ],
    "05": [
        "daniel.keller@stud.hs-hannover.de",
        "wendenburg.ludwig@stud.hs-hannover.de",
    ],
    "06": [
        "ayfer.celik@stud.hs-hannover.de",
        "hai-anh.nguyen@stud.hs-hannover.de",
        "finn-niklas.wermter@stud.hs-hannover.de",
    ],
    "07": [
        "ivan.chinskyi@stud.hs-hannover.de",
        "vladyslav.velychko@stud.hs-hannover.de",
    ],
    "08": [],
    "09": [],
    "10": [],
}

ohne_gruppe = [
    "andres-gabriel.castro-gonzales@stud.hs-hannover.de",
]

# Alle E-Mails sammeln
alle_emails = [e for members in gruppen.values() for e in members] + ohne_gruppe
print(f"{len(alle_emails)} User in {len([g for g in gruppen.values() if g])} aktiven Gruppen")

## 3. User über SCIM API anlegen

> Im Free Workspace können User **nicht** per SQL erstellt werden.
> Stattdessen wird die SCIM REST API genutzt.

In [ ]:
import requests
import json

SCIM_URL = f"{WORKSPACE_URL}/api/2.0/preview/scim/v2/Users"
HEADERS = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/scim+json",
    "Accept": "application/scim+json",
}


def create_user(email: str) -> dict:
    """Erstellt einen User über die SCIM API. Gibt das API-Ergebnis zurück."""
    # Vor- und Nachname aus der E-Mail ableiten
    local_part = email.split("@")[0]  # z.B. "selman.dogtas"
    name_parts = local_part.replace("-", " ").split(".")
    first_name = " ".join(p.capitalize() for p in name_parts[:-1]) if len(name_parts) > 1 else name_parts[0].capitalize()
    last_name = name_parts[-1].capitalize() if name_parts else ""

    payload = {
        "schemas": ["urn:ietf:params:scim:schemas:core:2.0:User"],
        "userName": email,
        "name": {
            "givenName": first_name,
            "familyName": last_name,
        },
        "emails": [{"type": "work", "value": email, "primary": True}],
        "active": True,
    }

    resp = requests.post(SCIM_URL, headers=HEADERS, json=payload)
    return {"email": email, "status": resp.status_code, "detail": resp.json()}


# Alle User anlegen
results = []
for email in alle_emails:
    result = create_user(email)
    status = result["status"]
    if status == 201:
        print(f"  ✅ {email} – angelegt")
    elif status == 409:
        print(f"  ⏭️  {email} – existiert bereits")
    else:
        print(f"  ❌ {email} – Fehler {status}: {result['detail']}")
    results.append(result)

created = sum(1 for r in results if r["status"] == 201)
existed = sum(1 for r in results if r["status"] == 409)
failed  = sum(1 for r in results if r["status"] not in (201, 409))
print(f"\n📊 Ergebnis: {created} neu, {existed} existierten bereits, {failed} fehlgeschlagen")

## 4. Kontrolle – vorhandene User auflisten

In [ ]:
resp = requests.get(SCIM_URL, headers=HEADERS, params={"count": 100})
users = resp.json().get("Resources", [])

print(f"Insgesamt {len(users)} User im Workspace:\n")
for u in sorted(users, key=lambda x: x.get("userName", "")):
    username = u.get("userName", "–")
    name = u.get("name", {})
    display = f"{name.get('givenName', '')} {name.get('familyName', '')}".strip()
    marker = " ← HSH" if "hs-hannover.de" in username else ""
    print(f"  {username:50s} {display}{marker}")

## 5. Gruppen anlegen und User zuweisen (SCIM API)

> Gruppen können im Free Workspace ebenfalls nicht per SQL angelegt werden.
> Wir nutzen die SCIM Groups API (`/api/2.0/preview/scim/v2/Groups`).

In [ ]:
GROUPS_URL = f"{WORKSPACE_URL}/api/2.0/preview/scim/v2/Groups"


def get_user_id(email: str) -> str | None:
    """User-ID anhand der E-Mail über SCIM API ermitteln."""
    resp = requests.get(SCIM_URL, headers=HEADERS,
                        params={"filter": f'userName eq "{email}"'})
    resources = resp.json().get("Resources", [])
    return resources[0]["id"] if resources else None


def create_group_with_members(group_name: str, member_emails: list[str]) -> dict:
    """Erstellt eine Gruppe und fügt die Mitglieder direkt hinzu."""
    # Prüfen ob Gruppe bereits existiert
    resp = requests.get(GROUPS_URL, headers=HEADERS,
                        params={"filter": f'displayName eq "{group_name}"'})
    existing = resp.json().get("Resources", [])

    # Mitglieder-IDs auflösen
    members = []
    for email in member_emails:
        uid = get_user_id(email)
        if uid:
            members.append({"value": uid})
        else:
            print(f"    ⚠️  User {email} nicht gefunden – übersprungen")

    if existing:
        # Gruppe existiert → Mitglieder per PATCH aktualisieren
        group_id = existing[0]["id"]
        if members:
            patch_payload = {
                "schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
                "Operations": [{
                    "op": "add",
                    "path": "members",
                    "value": members,
                }],
            }
            resp = requests.patch(f"{GROUPS_URL}/{group_id}",
                                  headers=HEADERS, json=patch_payload)
        return {"group": group_name, "status": "updated", "members": len(members)}

    else:
        # Gruppe neu anlegen
        payload = {
            "schemas": ["urn:ietf:params:scim:schemas:core:2.0:Group"],
            "displayName": group_name,
            "members": members,
        }
        resp = requests.post(GROUPS_URL, headers=HEADERS, json=payload)
        if resp.status_code == 201:
            return {"group": group_name, "status": "created", "members": len(members)}
        else:
            return {"group": group_name, "status": f"error ({resp.status_code})",
                    "detail": resp.json()}


# Gruppen anlegen und User zuweisen
for nr, emails in gruppen.items():
    group_name = f"bis242_gruppe{nr}"
    result = create_group_with_members(group_name, emails)
    status = result["status"]
    count = result.get("members", 0)
    if status in ("created", "updated"):
        print(f"  ✅ {group_name} – {status} ({count} Mitglieder)")
    else:
        print(f"  ❌ {group_name} – {result}")

print("\nVorhandene Gruppen:")
resp = requests.get(GROUPS_URL, headers=HEADERS, params={"count": 100})
for g in sorted(resp.json().get("Resources", []), key=lambda x: x.get("displayName", "")):
    name = g.get("displayName", "")
    if name.startswith("bis242_"):
        member_count = len(g.get("members", []))
        print(f"  {name} ({member_count} Mitglieder)")

## 6. Datenbanken anlegen

In [ ]:
for nr in [f"{i:02d}" for i in range(1, 11)]:
    spark.sql(f"CREATE DATABASE IF NOT EXISTS bis242_{nr} COMMENT 'BIS242 SS26 – Gruppe {nr}'")
    print(f"  ✅ bis242_{nr}")

print("\nVorhandene Datenbanken:")
spark.sql("SHOW DATABASES LIKE 'bis242_*'").show(truncate=False)

## 7. Owner der Datenbanken setzen

> **Einschränkung Community Edition:** SCIM-Gruppen werden nicht als Datenbank-Principals erkannt.
> Als Workaround wird jeweils das **erste Gruppenmitglied** als Owner gesetzt.
> Im Free Workspace haben ohnehin alle User Zugriff auf alle Datenbanken – die Trennung erfolgt per Konvention.

In [ ]:
for nr, emails in gruppen.items():
    db_name = f"bis242_{nr}"
    if not emails:
        print(f"  ⏭️  {db_name} – keine Mitglieder, Owner bleibt Admin")
        continue
    owner_email = emails[0]  # erstes Gruppenmitglied als Owner
    try:
        spark.sql(f"ALTER DATABASE {db_name} SET OWNER TO `{owner_email}`")
        print(f"  ✅ {db_name} → Owner: {owner_email}")
    except Exception as e:
        print(f"  ❌ {db_name} → {e}")

## 8. Aufräumen (Semesterende)

⚠️ **Nur bei Semesterende ausführen – löscht alle Daten unwiderruflich!**

In [ ]:
# ACHTUNG: Auskommentierung erst entfernen, wenn wirklich aufgeräumt werden soll!

# --- Datenbanken löschen ---
# for nr in [f"{i:02d}" for i in range(1, 11)]:
#     spark.sql(f"DROP DATABASE IF EXISTS bis242_{nr} CASCADE")
#     print(f"  🗑️  bis242_{nr} gelöscht")

# --- User deaktivieren ---
# for email in alle_emails:
#     # User-ID ermitteln
#     resp = requests.get(SCIM_URL, headers=HEADERS,
#                         params={"filter": f'userName eq "{email}"'})
#     resources = resp.json().get("Resources", [])
#     if resources:
#         user_id = resources[0]["id"]
#         patch = {"schemas": ["urn:ietf:params:scim:api:messages:2.0:PatchOp"],
#                  "Operations": [{"op": "replace", "path": "active", "value": False}]}
#         requests.patch(f"{SCIM_URL}/{user_id}", headers=HEADERS, json=patch)
#         print(f"  🔒 {email} deaktiviert")

print("Aufräum-Block ist auskommentiert. Kommentierung entfernen um auszuführen.")